# 03 回测示例

策略统一通过 quant.strategy.run_backtest(name, symbol=..., cash=...) 调用，
返回 BacktestResult（收益率 / 最大回撤 / 交易次数 / 胜率）。

命令行等价形式：quant backtest --strategy ma_obv --symbol 512480

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from quant.strategy import list_strategies, run_backtest

for info in list_strategies():
    print("%-18s 默认 %-8s %s" % (info.name, info.default_symbol, info.description))
    for param in info.params:
        print("      %-14s %s (默认 %s)" % (param["name"], param["label"], param["default"]))

## MA + OBV（ETF）

In [ ]:
result = run_backtest("ma_obv", symbol="512480", cash=100000, commission=0.001, ma_period=20)
print(result.report())

import matplotlib.pyplot as plt

if result.equity_curve is not None:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(result.equity_curve["date"], result.equity_curve["value"], label="策略")
    if result.benchmark_curve is not None:
        ax.plot(result.benchmark_curve["date"], result.benchmark_curve["value"],
                label="买入持有", linestyle="--")
    ax.set_title("净值曲线")
    ax.legend()
    plt.tight_layout()
    plt.show()

pd.DataFrame(result.trade_list).head(10)

## 缩量买入（个股）

In [ ]:
result2 = run_backtest("volume_breakout", symbol="000001", cash=100000,
                       volume_ratio=0.4, stop_loss=0.05, take_profit=0.3)
print(result2.report())
pd.DataFrame(result2.trade_list).head(10)

## 批量对比

数据库里没有日线的标的会抛 FileNotFoundError，这里捕获后记录原因。

In [ ]:
rows = []
for symbol in ["510880", "512480", "159611", "159928"]:
    try:
        r = run_backtest("ma_obv", symbol=symbol, cash=100000)
        rows.append({"symbol": symbol, **r.metrics()})
    except Exception as exc:
        rows.append({"symbol": symbol, "error": str(exc)[:40]})

pd.DataFrame(rows)

## 画图

run_backtest(..., plot=True) 会调用 backtrader 的绘图。

In [ ]:
# run_backtest("ma_obv", symbol="512480", plot=True)
print("取消注释即可画图")